# Electronics Store — Data Analytics Project

## Task 1 — Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

In [ ]:
df = pd.read_excel("Electronics_Store_Dataset.xlsx")

In [ ]:
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")

print("\n" + "=" * 50)
print("COLUMN NAMES & DATA TYPES")
print("=" * 50)
print(df.dtypes.to_string())

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
df.head()

In [ ]:
print("=" * 50)
print("BASIC STATISTICS (Numeric Columns)")
print("=" * 50)
df.describe().round(2)

In [ ]:
print("=" * 50)
print("UNIQUE VALUES PER COLUMN")
print("=" * 50)
for col in df.columns:
    n = df[col].nunique()
    sample = list(df[col].dropna().unique()[:4])
    print(f"  {col:<20} {n:>4} unique  →  {sample}")

## Task 2 — Data Cleaning

In [ ]:
print("Missing values BEFORE cleaning:")
print(df.isnull().sum().to_string())

In [ ]:
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')
df.head(10)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
print("\nMissing values AFTER cleaning:")
print(df.isnull().sum().to_string())

In [ ]:
print(f"Negative Quantity rows : {(df['Quantity'] <= 0).sum()}")
print(f"Negative UnitPrice rows: {(df['UnitPrice'] <= 0).sum()}")
print(f"Negative TotalPrice rows: {(df['TotalPrice'] <= 0).sum()}")

In [ ]:
df['CalcTotal'] = (df['Quantity'] * df['UnitPrice']).round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)
mismatch = (df['CalcTotal'] != df['TotalPrice']).sum()
print(f"TotalPrice mismatches  : {mismatch}")
df.drop(columns=['CalcTotal'], inplace=True)

In [ ]:
print(" Dataset is clean and ready for analysis!")
print(f"Final shape: {df.shape}")

## Task 3 — Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 50)
print("KEY STATISTICS")
print("=" * 50)
print(f"Total Revenue      : ${df['TotalPrice'].sum():,.2f}")
print(f"Average Order Value: ${df['TotalPrice'].mean():,.2f}")
print(f"Median Order Value : ${df['TotalPrice'].median():,.2f}")
print(f"Max Order Value    : ${df['TotalPrice'].max():,.2f}")
print(f"Min Order Value    : ${df['TotalPrice'].min():,.2f}")

print("\nOrders by Status:")
print(df['OrderStatus'].value_counts().to_string())

print("\nTop Products by Order Count:")
print(df['Product'].value_counts().to_string())

print("\nAverage Order Value by Product:")
print(df.groupby('Product')['TotalPrice'].mean().round(2).sort_values(ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['TotalPrice'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Order Total Price', fontsize=13)
axes[0].set_xlabel('Total Price ($)')
axes[0].set_ylabel('Number of Orders')

axes[1].boxplot(df['TotalPrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Box Plot — Detecting Outliers in TotalPrice', fontsize=13)
axes[1].set_ylabel('Total Price ($)')

plt.tight_layout()
plt.show()

Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['TotalPrice'] < Q1 - 1.5*IQR) | (df['TotalPrice'] > Q3 + 1.5*IQR)]
print(f"Outliers detected (IQR method): {len(outliers)} orders")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

product_counts   = df['Product'].value_counts()
product_revenue  = df.groupby('Product')['TotalPrice'].sum().sort_values(ascending=False)

axes[0].bar(product_counts.index, product_counts.values, color='cornflowerblue', edgecolor='white')
axes[0].set_title('Number of Orders by Product', fontsize=13)
axes[0].set_xlabel('Product'); axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(product_revenue.index, product_revenue.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Total Revenue by Product', fontsize=13)
axes[1].set_xlabel('Product'); axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

status_counts = df['OrderStatus'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'), startangle=140)
axes[0].set_title('Order Status Distribution', fontsize=13)

ref_counts = df['ReferralSource'].value_counts()
axes[1].bar(ref_counts.index, ref_counts.values, color=sns.color_palette('Set2'))
axes[1].set_title('Customers by Referral Source', fontsize=13)
axes[1].set_xlabel('Source'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
numeric_cols = df[['Quantity', 'UnitPrice', 'TotalPrice', 'ItemsInCart']]
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## Task 4 — Advanced Visualizations & Trend Analysis

In [ ]:
monthly = df.groupby(['Year', 'Month'])['TotalPrice'].sum().reset_index()
monthly['Period'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))
monthly = monthly.sort_values('Period')

plt.figure(figsize=(13, 4))
plt.plot(monthly['Period'], monthly['TotalPrice'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(monthly['Period'], monthly['TotalPrice'], alpha=0.15, color='steelblue')
plt.title('Monthly Revenue Trend', fontsize=14)
plt.xlabel('Month'); plt.ylabel('Revenue ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(['Product', 'OrderStatus']).size().unstack(fill_value=0)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlGnBu', linewidths=0.5)
plt.title('Order Count — Product × Status', fontsize=13)
plt.xlabel('Order Status'); plt.ylabel('Product')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pay_rev = df.groupby('PaymentMethod')['TotalPrice'].sum().sort_values(ascending=False)
axes[0].bar(pay_rev.index, pay_rev.values, color=sns.color_palette('muted'), edgecolor='white')
axes[0].set_title('Revenue by Payment Method', fontsize=13)
axes[0].set_xlabel('Payment Method'); axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=20)

coupon_counts = df['CouponCode'].value_counts()
axes[1].bar(coupon_counts.index, coupon_counts.values,
            color=sns.color_palette('tab10', len(coupon_counts)), edgecolor='white')
axes[1].set_title('Orders by Coupon Code', fontsize=13)
axes[1].set_xlabel('Coupon Code'); axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

avg_ref = df.groupby('ReferralSource')['TotalPrice'].mean().sort_values(ascending=False)
axes[0].barh(avg_ref.index, avg_ref.values, color='mediumpurple', edgecolor='white')
axes[0].set_title('Avg Order Value by Referral Source', fontsize=13)
axes[0].set_xlabel('Avg Total Price ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

df.boxplot(column='TotalPrice', by='Product', ax=axes[1],
           patch_artist=True, figsize=(13,4))
axes[1].set_title('TotalPrice Distribution by Product', fontsize=13)
axes[1].set_xlabel('Product'); axes[1].set_ylabel('Total Price ($)')
plt.suptitle('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Task 5 — Machine Learning Models

In [ ]:
df_model = df.copy()
le = LabelEncoder()
for col in ['Product', 'PaymentMethod', 'ReferralSource', 'CouponCode', 'OrderStatus']:
    df_model[col + '_enc'] = le.fit_transform(df_model[col])

features = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'ReferralSource_enc', 'CouponCode_enc']

X = df_model[features]
print("Features used:", features)
print("Sample encoded data:")
X.head()

In [ ]:
y_reg = df_model['TotalPrice']

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred_reg = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_reg)
r2  = r2_score(y_test, y_pred_reg)

print("=" * 45)
print("  MODEL 1: Linear Regression — TotalPrice")
print("=" * 45)
print(f"  Mean Absolute Error : ${mae:.2f}")
print(f"  R² Score            : {r2:.4f}  ({r2*100:.1f}% variance explained)")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_reg, alpha=0.4, color='steelblue', edgecolors='white', s=40)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Total Price ($)')
plt.ylabel('Predicted Total Price ($)')
plt.title('Regression — Actual vs Predicted TotalPrice', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_model['IsDelivered'] = (df_model['OrderStatus'] == 'Delivered').astype(int)
y_clf = df_model['IsDelivered']

X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y_clf, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train2_sc = scaler.fit_transform(X_train2)
X_test2_sc  = scaler.transform(X_test2)

clf_model = LogisticRegression(max_iter=300, random_state=42)
clf_model.fit(X_train2_sc, y_train2)
y_pred_clf = clf_model.predict(X_test2_sc)

acc = accuracy_score(y_test2, y_pred_clf)

print("=" * 45)
print("  MODEL 2: Logistic Regression — Delivered?")
print("=" * 45)
print(f"  Accuracy: {acc*100:.2f}%")
print()
print(classification_report(y_test2, y_pred_clf, target_names=['Not Delivered', 'Delivered'], zero_division=0))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test2, y_pred_clf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Delivered', 'Delivered'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Delivered Prediction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({
    'Feature'    : features,
    'Coefficient': reg_model.coef_
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(8, 4))
colors = ['mediumseagreen' if c > 0 else 'salmon' for c in coef_df['Coefficient']]
plt.bar(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.title('Feature Importance — Linear Regression Coefficients', fontsize=13)
plt.xlabel('Feature'); plt.ylabel('Coefficient Value')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()